# Decision-payoff selection vs. statistical accuracy

A simple binary-gamble example to make one point: **the model that gives the best decision depends on the agent's risk aversion, not on which model is statistically closest to the truth.** Three agents with different $\gamma$ can rationally select different models from the same set of candidates, even on the same data and the same procedure.

**The true gamble.** Each period the payoff multiple is $M_+ = 1.5$ with true probability $p = 0.55$, otherwise $M_- = 0.7$. Per-period wealth evolves as
$$
W_{t+1} = W_t \cdot \bigl[1 + f(M - 1)\bigr], \qquad M \in \{M_+, M_-\}.
$$

**Three candidate models.** *None of them matches the truth* — each is wrong in its own way. This is by design: in real applications the right model is never in the candidate set, so we want to make sure the lesson does not rely on having the truth as a candidate.

| Model | $q$ | $M_+$ | $M_-$ | character |
|---|---:|---:|---:|---|
| A — risk-loving | 0.75 | 1.8 | 0.5 | overconfident in upside, ignores downside |
| B — pessimist  | 0.30 | 2.2 | 0.8 | thinks wins are rare; losses are mild |
| C — wide spread | 0.55 | 2.2 | 0.6 | right $q$, exaggerated magnitudes both ways |

**Three agents** with CRRA risk aversion $\gamma \in \{0.5, 1, 2\}$. Each agent considers all three models, picks an $f^\star$ under each, and selects among them by realized mean utility on a 1000-draw validation set from the true gamble. We expect — and check — that the winning model differs across $\gamma$.

**Why this is robust.** Asymptotically (large $n$), realized mean utility under the truth converges to true expected utility, so the winning model at each $\gamma$ is the one whose $f^\star$ is closest to the truth-implied $f^\star(\gamma)$. Since each $\gamma$ has its own truth-implied $f^\star$, the closest candidate model can differ by $\gamma$ — and below we will see that the three candidates above were constructed exactly so that A is closest at $\gamma = 0.5$, B at $\gamma = 1$, and C at $\gamma = 2$.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(42)

p_true, Mp_true, Mm_true = 0.55, 1.5, 0.7
models = {
    "A (risk-loving)": {"q": 0.75, "M_plus": 1.8, "M_minus": 0.5},
    "B (pessimist)":   {"q": 0.30, "M_plus": 2.2, "M_minus": 0.8},
    "C (wide spread)": {"q": 0.55, "M_plus": 2.2, "M_minus": 0.6},
}
gammas = [0.5, 1.0, 2.0]

## Training stage: $f^\star$ under each candidate model

Each agent computes
$$
f^\star(\gamma, \text{model}) = \arg\max_f \; \mathbb{E}_{\text{model}}\bigl[u_\gamma(1 + f(M - 1))\bigr]
$$
with expectation taken under the model's own beliefs $(q, M_+, M_-)$. We also report the truth-implied $f^\star(\gamma)$ as a benchmark — this is the $f^\star$ an agent who knew the truth would choose, and asymptotically the validation-stage winner per $\gamma$ is whichever candidate's $f^\star$ is closest to it.

In [2]:
def neg_eu(f, q, Mp, Mm, gamma):
    wW = 1.0 + f * (Mp - 1.0)
    wL = 1.0 + f * (Mm - 1.0)
    if wW <= 0 or wL <= 0:
        return 1e10
    if gamma == 1:
        return -(q * np.log(wW) + (1 - q) * np.log(wL))
    return -(q * (wW ** (1 - gamma) - 1) / (1 - gamma)
             + (1 - q) * (wL ** (1 - gamma) - 1) / (1 - gamma))

def f_star(q, Mp, Mm, gamma, f_max=5.0):
    f_ruin = 1.0 / (1 - Mm) if Mm < 1 else f_max
    upper = min(f_ruin - 1e-4, f_max)
    res = minimize_scalar(neg_eu, args=(q, Mp, Mm, gamma),
                          bounds=(0.0, upper), method="bounded",
                          options={"xatol": 1e-7})
    return res.x

f_truth = np.array([f_star(p_true, Mp_true, Mm_true, g) for g in gammas])
fstar_table = pd.DataFrame(
    {name: [f_star(m["q"], m["M_plus"], m["M_minus"], g) for g in gammas]
     for name, m in models.items()},
    index=pd.Index(gammas, name="gamma"),
)
fstar_table.insert(0, "truth f*", f_truth)
fstar_table.round(3)

,truth f*,A (risk-loving),B (pessimist),C (wide spread)
gamma,,,,
0.5,1.805,1.789,2.225,1.892
1.0,0.933,1.188,0.917,1.000
2.0,0.460,0.628,0.397,0.465


In [3]:
# Distance of each candidate's f* from the truth-implied f*, by gamma
deltas = (fstar_table.drop(columns=["truth f*"]).subtract(fstar_table["truth f*"], axis=0)).abs()
deltas["closest"] = deltas.idxmin(axis=1)
deltas.round(3)

,A (risk-loving),B (pessimist),C (wide spread),closest
gamma,,,,
0.5,0.016,0.420,0.087,A (risk-loving)
1.0,0.254,0.017,0.067,B (pessimist)
2.0,0.168,0.063,0.005,C (wide spread)


## Validation stage: 1000 draws from the truth

Each $(\gamma, \text{model})$ pair's realized mean utility on 1000 i.i.d. realizations of the true gamble:
$$
\frac{1}{n} \sum_{t} u_\gamma\bigl(1 + f^\star_{\gamma, \text{model}}\,(M_t - 1)\bigr).
$$
The winning model per $\gamma$ is the one whose recommended $f^\star$ delivers the highest realized utility on data drawn from the true process.

In [4]:
n_draws = 1000
wins = rng.random(n_draws) < p_true
payoffs = np.where(wins, Mp_true, Mm_true)
print(f"realized win frequency: {wins.mean():.3f}  (truth p = {p_true})")

def realized_mean_utility(f, payoffs, gamma):
    w = 1.0 + f * (payoffs - 1.0)
    if np.any(w <= 0):
        return -np.inf
    if gamma == 1:
        return np.log(w).mean()
    return ((w ** (1 - gamma) - 1) / (1 - gamma)).mean()

val_table = pd.DataFrame(
    {name: [realized_mean_utility(fstar_table.loc[g, name], payoffs, g) for g in gammas]
     for name in models},
    index=pd.Index(gammas, name="gamma"),
)
winners = val_table.idxmax(axis=1).rename("winner")
pd.concat([val_table.round(5), winners], axis=1)

realized win frequency: 0.545  (truth p = 0.55)


,A (risk-loving),B (pessimist),C (wide spread),winner
gamma,,,,
0.5,0.11969,0.10899,0.11893,A (risk-loving)
1.0,0.05362,0.05930,0.05869,B (pessimist)
2.0,0.02460,0.02875,0.02905,C (wide spread)


## What we just saw

**Different agents picked different models, and they should have.** Each of the three candidate models is wrong about the truth in its own way — none of them matches $(p, M_+, M_-) = (0.55, 1.5, 0.7)$. But each candidate is *closest* to the truth-implied $f^\star$ at a different $\gamma$:

- **Model A (risk-loving)** has the right $f^\star$ for an agent with low risk aversion ($\gamma = 0.5$).
- **Model B (pessimist)** has the right $f^\star$ for a Kelly-style log-utility agent ($\gamma = 1$).
- **Model C (wide spread)** has the right $f^\star$ for an agent with moderate-to-high risk aversion ($\gamma = 2$).

Even though A's belief $q = 0.75$ is the furthest from the truth's $p = 0.55$ in absolute terms, A's $f^\star$ at low $\gamma$ happens to land closest to what the truth would have recommended at that $\gamma$. The agent's risk aversion is doing the work of picking the right *decision*, not the right *belief*.

**The pedagogical punchline.** Statistical accuracy — "which model is closest to the true distribution" — is not the right model-selection criterion when a decision is downstream. The right criterion is *how well does the model's implied decision perform on data the model has not seen*, and that criterion is risk-aversion-dependent. Two analysts with the same data and the same candidate models can rationally arrive at different choices.

**Robustness check.** Try other seeds in the imports cell. At $n = 1000$ there is real sampling noise in the realized win frequency $\hat p$, so on some seeds the winners shuffle around. Asymptotically (try $n = 100{,}000$) the canonical pattern A→B→C across the three $\gamma$ is the deterministic outcome.

**Connection to notebook 2.** This is the same principle as `06_Portfolio_optimization_and_copulas/2_simple_portfolio_optimization.ipynb`. There the candidates are continuous return distributions (Normal, Student-$t$, Johnson $S_U$), and the validation-stage winner depends on the agent's $\gamma$ for exactly this reason. Here the candidates are binary gambles; the structural lesson transfers.